# Chapter 01. 베스트셀러 데이터 이해와 기본 전처리

교보문고 온라인 베스트셀러 Excel을 불러와 구조를 확인하고, 이후 텍스트 분석에 필요한 8개 컬럼을 정리한다.

실습 기준: [과정 소개](https://blog.naver.com/dev-dog/224415732221) · [Chapter 01 가이드](https://blog.naver.com/dev-dog/224415736311)

## 분석 흐름

Excel 로딩 → 구조 확인 → 컬럼 선택·이름 정리 → 결측·자료형·중복 점검 → 저장 → 재로딩 검증

In [1]:
from pathlib import Path
import warnings
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", message="Workbook contains no default style")
pd.set_option("display.max_columns", 20)

BASE_DIR = Path.cwd()
if not (BASE_DIR / "교보문고_온라인_베스트셀러_상품리스트.xlsx").exists():
    candidate = BASE_DIR / "book-text-ml"
    if (candidate / "교보문고_온라인_베스트셀러_상품리스트.xlsx").exists():
        BASE_DIR = candidate

DATA_PATH = BASE_DIR / "교보문고_온라인_베스트셀러_상품리스트.xlsx"
OUTPUT_PATH = BASE_DIR / "book_bestseller_clean.csv"
assert DATA_PATH.exists(), f"원본 파일을 찾을 수 없습니다: {DATA_PATH}"

print("작업 폴더:", BASE_DIR.resolve())
print("원본 파일:", DATA_PATH.name)

작업 폴더: /home/gem/research/4. 학교 수업/LLM 수업/llm-data-analysis-study/book-text-ml
원본 파일: 교보문고_온라인_베스트셀러_상품리스트.xlsx


## 1. Excel 불러오기와 구조 확인

전처리 전에 `shape`, `columns`, `dtypes`, 결측치와 일부 행을 함께 확인한다.

In [2]:
df = pd.read_excel(DATA_PATH)

print("원본 크기:", df.shape)
print("컬럼:", df.columns.tolist())
print("\n자료형:")
display(df.dtypes.rename("dtype").to_frame())
print("결측치:")
display(df.isna().sum().rename("결측").to_frame())
display(df.head())

원본 크기: (993, 13)
컬럼: ['순위', '상품코드', '판매상품 ID', '상품명', '정가', '판매가', '할인율', '적립율', '적립예정포인트', '인물', '출판사', '발행(출시)일자', '분야']

자료형:


,dtype
순위,int64
상품코드,str
판매상품 ID,str
상품명,str
정가,str
판매가,str
할인율,str
적립율,str
적립예정포인트,str
인물,str


결측치:


,결측
순위,0
상품코드,0
판매상품 ID,0
상품명,0
정가,0
판매가,0
할인율,0
적립율,0
적립예정포인트,0
인물,6


,순위,상품코드,판매상품 ID,상품명,정가,판매가,할인율,적립율,적립예정포인트,인물,출판사,발행(출시)일자,분야
0,1,9791173713422,S000221773365,판다월드에 놀러 오세요!,"16,800","15,120",10%,5%,840,말하는동물원 뿌빠TV,서울문화사,20261125.0,어린이(초등)
1,2,9791167743428,S000221888549,공부력 2027,"24,000","21,600",10%,5%,"1,200",김영민,어크로스,20261127.0,인문
2,3,9788931484472,S000221881792,독자생존,"18,800","16,920",10%,5%,940,이경희 외,영진닷컴,20261120.0,인문
3,4,9791141691370,S000221505184,가호,"18,500","16,650",10%,5%,920,무라카미 하루키,문학동네,20261021.0,소설
4,5,9788901301129,S000221957683,겨울 선물의 전설,"20,000","18,000",10%,5%,"1,000",이지은,웅진주니어,20261117.0,유아(0~7세)


**결과 관찰:** 원본은 993행, 13열이며 저자 역할의 `인물`, `출판사`, `발행(출시)일자`, `분야`에 결측치가 있다. `판매가`는 쉼표가 포함된 문자열이고 발행일은 숫자형으로 읽혔다.

**나의 판단:** 단어 분석에 필요한 컬럼만 남기되, 결측·중복을 먼저 확인하고 자동 삭제하지 않는다.

## 2. 필요한 컬럼 선택과 이름 정리

In [3]:
selected_columns = [
    "순위", "판매상품 ID", "상품명", "판매가",
    "인물", "출판사", "발행(출시)일자", "분야",
]

df_books = df[selected_columns].copy().rename(columns={
    "판매상품 ID": "판매상품ID",
    "인물": "저자",
    "발행(출시)일자": "발행일",
})

print("선택 후 크기:", df_books.shape)
print("정리된 컬럼:", df_books.columns.tolist())
display(df_books.head())

선택 후 크기: (993, 8)
정리된 컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']


,순위,판매상품ID,상품명,판매가,저자,출판사,발행일,분야
0,1,S000221773365,판다월드에 놀러 오세요!,"15,120",말하는동물원 뿌빠TV,서울문화사,20261125.0,어린이(초등)
1,2,S000221888549,공부력 2027,"21,600",김영민,어크로스,20261127.0,인문
2,3,S000221881792,독자생존,"16,920",이경희 외,영진닷컴,20261120.0,인문
3,4,S000221505184,가호,"16,650",무라카미 하루키,문학동네,20261021.0,소설
4,5,S000221957683,겨울 선물의 전설,"18,000",이지은,웅진주니어,20261117.0,유아(0~7세)


## 3. 결측치 확인과 처리

가이드 기준에 따라 `저자`는 `미상`, `분야`는 `미분류`로 보존한다. 출판사와 발행일 결측은 분석 목적상 임의 값을 만들지 않는다.

In [4]:
missing_before = df_books.isna().sum().rename("처리 전 결측")
df_books["저자"] = df_books["저자"].fillna("미상")
df_books["분야"] = df_books["분야"].fillna("미분류")
missing_after = df_books.isna().sum().rename("처리 후 결측")

missing_comparison = pd.concat([missing_before, missing_after], axis=1)
display(missing_comparison)

,처리 전 결측,처리 후 결측
순위,0,0
판매상품ID,0,0
상품명,0,0
판매가,0,0
저자,6,0
출판사,7,7
발행일,6,6
분야,52,0


## 4. 판매가와 발행일 자료형 변환

변환 전 값이 존재했는데 변환 후 결측이 된 행을 별도로 세어, 변환 실패가 숨어 있지 않은지 확인한다.

In [5]:
raw_price = df_books["판매가"].copy()
price_text = raw_price.astype("string").str.replace(",", "", regex=False)
df_books["판매가"] = pd.to_numeric(price_text, errors="coerce")
price_conversion_failures = int((raw_price.notna() & df_books["판매가"].isna()).sum())

print("판매가 dtype:", df_books["판매가"].dtype)
print("판매가 신규 변환 실패:", price_conversion_failures)
display(df_books["판매가"].describe().to_frame().T)

판매가 dtype: Int64
판매가 신규 변환 실패: 0


,count,mean,std,min,25%,50%,75%,max
판매가,993.0,18383.434038,10373.082702,1500.0,13500.0,16920.0,19800.0,129600.0


In [6]:
raw_date = df_books["발행일"].copy()
date_text = pd.to_numeric(raw_date, errors="coerce").astype("Int64").astype("string")
df_books["발행일"] = pd.to_datetime(date_text, format="%Y%m%d", errors="coerce")
date_conversion_failures = int((raw_date.notna() & df_books["발행일"].isna()).sum())

print("발행일 dtype:", df_books["발행일"].dtype)
print("원래 결측:", int(raw_date.isna().sum()))
print("신규 변환 실패:", date_conversion_failures)
print("날짜 범위:", df_books["발행일"].min().date(), "~", df_books["발행일"].max().date())
display(df_books[["발행일"]].head())

발행일 dtype: datetime64[us]
원래 결측: 6
신규 변환 실패: 0
날짜 범위: 1998-02-01 ~ 2027-01-10


,발행일
0,2026-11-25
1,2026-11-27
2,2026-11-20
3,2026-10-21
4,2026-11-17


## 5. 문자열 공백과 중복 점검

In [7]:
string_columns = ["상품명", "저자", "출판사", "분야"]
for column in string_columns:
    df_books[column] = df_books[column].astype("string").str.strip()

blank_counts = {
    column: int(df_books[column].fillna("").eq("").sum())
    for column in string_columns
}
print("빈 문자열 개수:", blank_counts)

빈 문자열 개수: {'상품명': 0, '저자': 0, '출판사': 7, '분야': 0}


In [8]:
duplicate_summary = pd.Series({
    "전체 행 중복": int(df_books.duplicated().sum()),
    "판매상품ID 중복": int(df_books.duplicated(subset="판매상품ID").sum()),
    "상품명 중복": int(df_books.duplicated(subset="상품명").sum()),
}, name="건수").to_frame()

display(duplicate_summary)
print("상품명 중복은 종이책·전자책 또는 서로 다른 판본일 수 있으므로 삭제하지 않는다.")

,건수
전체 행 중복,0
판매상품ID 중복,0
상품명 중복,34


상품명 중복은 종이책·전자책 또는 서로 다른 판본일 수 있으므로 삭제하지 않는다.


## 6. 최종 검증과 저장

In [9]:
validation = pd.DataFrame([
    ["행 수 유지", len(df), len(df_books), len(df) == len(df_books)],
    ["최종 컬럼 수", 8, len(df_books.columns), len(df_books.columns) == 8],
    ["판매가 변환 실패", 0, price_conversion_failures, price_conversion_failures == 0],
    ["발행일 신규 변환 실패", 0, date_conversion_failures, date_conversion_failures == 0],
    ["판매상품ID 결측", 0, int(df_books["판매상품ID"].isna().sum()), df_books["판매상품ID"].notna().all()],
    ["판매상품ID 중복", 0, int(df_books.duplicated("판매상품ID").sum()), not df_books.duplicated("판매상품ID").any()],
], columns=["검증", "기대값", "실제값", "PASS"])

display(validation)
assert validation["PASS"].all(), "전처리 검증에 실패했습니다."
display(df_books.head())

,검증,기대값,실제값,PASS
0,행 수 유지,993,993,True
1,최종 컬럼 수,8,8,True
2,판매가 변환 실패,0,0,True
3,발행일 신규 변환 실패,0,0,True
4,판매상품ID 결측,0,0,True
5,판매상품ID 중복,0,0,True


,순위,판매상품ID,상품명,판매가,저자,출판사,발행일,분야
0,1,S000221773365,판다월드에 놀러 오세요!,15120,말하는동물원 뿌빠TV,서울문화사,2026-11-25,어린이(초등)
1,2,S000221888549,공부력 2027,21600,김영민,어크로스,2026-11-27,인문
2,3,S000221881792,독자생존,16920,이경희 외,영진닷컴,2026-11-20,인문
3,4,S000221505184,가호,16650,무라카미 하루키,문학동네,2026-10-21,소설
4,5,S000221957683,겨울 선물의 전설,18000,이지은,웅진주니어,2026-11-17,유아(0~7세)


In [10]:
df_books.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

reloaded = pd.read_csv(OUTPUT_PATH, encoding="utf-8-sig")
reload_validation = pd.DataFrame([
    ["파일 생성", True, OUTPUT_PATH.exists()],
    ["행 수", len(df_books), len(reloaded)],
    ["컬럼 일치", True, reloaded.columns.tolist() == df_books.columns.tolist()],
    ["판매가 재로딩 가능", True, pd.to_numeric(reloaded["판매가"], errors="coerce").notna().all()],
    ["발행일 재변환 실패 0", 0, int(reloaded["발행일"].notna().sum() - pd.to_datetime(reloaded["발행일"], errors="coerce").notna().sum())],
], columns=["검증", "기대값", "실제값"])
reload_validation["PASS"] = reload_validation["기대값"] == reload_validation["실제값"]

display(reload_validation)
assert reload_validation["PASS"].all(), "저장 파일 재검증에 실패했습니다."
print("저장 완료:", OUTPUT_PATH.name, OUTPUT_PATH.stat().st_size, "bytes")

,검증,기대값,실제값,PASS
0,파일 생성,True,True,True
1,행 수,993,993,True
2,컬럼 일치,True,True,True
3,판매가 재로딩 가능,True,True,True
4,발행일 재변환 실패 0,0,0,True


저장 완료: book_bestseller_clean.csv 112361 bytes


## 확인 문제 답변

1. **왜 필요한 컬럼만 선택했는가?** 제목 텍스트 분석·분류·추천에 필요한 식별자, 제목, 가격, 서지·분야 정보에 분석 범위를 맞추기 위해서다.
2. **판매가가 문자열이면 어떤 문제가 생기는가?** 합계·평균·범위 같은 수치 계산과 비교가 정확히 동작하지 않을 수 있다.
3. **결측 행을 바로 모두 삭제하면 안 되는 이유는?** 저자나 분야가 비어 있어도 제목 빈도 분석에는 사용할 수 있으므로 분석 목적에 따라 처리해야 한다.
4. **같은 상품명이 바로 오류가 아닌 이유는?** 종이책·전자책, 출판사나 판본이 다른 별도 상품일 수 있기 때문이다.
5. **코드가 실행되면 분석이 끝난 것인가?** 아니다. 행 수, 자료형, 결측·중복, 변환 실패와 저장 결과를 다시 검증해야 한다.

## 결과 정리

- **관찰:** 993행에서 필요한 8개 컬럼을 선택했다. 저자 결측 6건과 분야 결측 52건은 각각 `미상`, `미분류`로 보존했다. 판매가와 발행일의 신규 변환 실패는 모두 0건이었다. 전체 행·판매상품ID 중복은 0건이며 상품명 중복은 34건이다.
- **해석과 판단:** 같은 상품명은 종이책·전자책 또는 판본 차이일 수 있어 삭제하지 않았다. 출판사 7건과 발행일 6건의 결측도 제목 빈도 분석에는 사용할 수 있으므로 유지했다.
- **한계:** 이 데이터는 2026-11-20에 내려받은 온라인 베스트셀러 스냅샷이므로 다른 날짜에 내려받은 목록과 결과가 달라질 수 있다.
- **다음 단계:** `book_bestseller_clean.csv`의 `상품명`을 대상으로 형태소 분석과 단어 빈도 분석을 진행한다.